<a href="https://colab.research.google.com/github/souleater369/Local-IOT-/blob/main/IOTT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import time
import os
import joblib
import io
import warnings
import datetime
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

def run_v7_1_academic_benchmark():
    print("Initializing V7.1 Master Academic Edge-Hardware Benchmark...\n")
    np.random.seed(42)

    # ==========================================
    # 1. DATA LOADING & DEFENSIVE CLEANING
    # ==========================================
    file_names = [
        'UNSW_2018_IoT_Botnet_Full5pc_1.csv',
        'UNSW_2018_IoT_Botnet_Full5pc_2.csv',
        'UNSW_2018_IoT_Botnet_Full5pc_3.csv',
        'UNSW_2018_IoT_Botnet_Full5pc_4.csv'
    ]

    print("Loading all dataset chunks. This may take a moment...")
    dataframes = [pd.read_csv(f, low_memory=False) for f in file_names if os.path.exists(f)]

    if not dataframes:
        raise ValueError("CRITICAL: No dataset chunks found. Check your filenames in this directory.")

    dataset = pd.concat(dataframes, ignore_index=True)
    print(f"Successfully stitched {len(dataframes)} files. Total raw packets: {len(dataset):,}\n")

    features = ['mean', 'stddev', 'bytes', 'seq']
    required = features + ['attack']

    missing = [c for c in required if c not in dataset.columns]
    if missing:
        raise ValueError(f"CRITICAL: Missing columns: {missing}")

    # Drop rows with any missing data
    dataset = dataset.dropna(subset=features)
    for col in features:
        dataset[col] = pd.to_numeric(dataset[col], errors='coerce')

    dataset['attack'] = pd.to_numeric(dataset['attack'], errors='coerce')
    dataset = dataset.dropna(subset=required)
    dataset['attack'] = dataset['attack'].astype(int)

    X = dataset[features]
    y = dataset['attack']

    # ==========================================
    # 2. NESTED EVALUATION (Outer Holdout Split)
    # ==========================================
    X_train_outer, X_test, y_train_outer, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    print("--- Traffic Distribution (Outer Split) ---")
    print(f"Training Set : Safe ({len(y_train_outer[y_train_outer==0]):,}) | Attack ({len(y_train_outer[y_train_outer==1]):,})")
    print(f"Holdout Set  : Safe ({len(y_test[y_test==0]):,}) | Attack ({len(y_test[y_test==1]):,})\n")

    # ==========================================
    # 3. ROBUST BALANCER FUNCTION
    # ==========================================
    def balance_data(X_data, y_data):
        train_data = pd.concat([X_data, y_data], axis=1)
        safe = train_data[train_data['attack'] == 0]
        attack = train_data[train_data['attack'] == 1]

        if len(attack) > len(safe) and len(safe) > 0:
            attack_down = resample(attack, replace=False, n_samples=len(safe), random_state=42)
            balanced = pd.concat([safe, attack_down])
        elif len(safe) > len(attack) and len(attack) > 0:
            safe_down = resample(safe, replace=False, n_samples=len(attack), random_state=42)
            balanced = pd.concat([safe_down, attack])
        else:
            balanced = train_data.copy()

        balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        return balanced[features], balanced['attack']

    # Balance the full outer training set for the final model build
    print("Balancing training data to ensure fair evaluation...")
    X_train_bal, y_train_bal = balance_data(X_train_outer, y_train_outer)
    X_test_np = X_test.values
    test_batch = X_test_np[:100]

    # ==========================================
    # 4. ARCHITECTURE DEFINITIONS
    # ==========================================
    models = {
        "DNN (MLP)": Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=100, random_state=42))
        ]),
        "Random Forest": RandomForestClassifier(criterion='entropy', max_depth=5, n_estimators=100, random_state=42),
        "V7.1 Edge Shield": DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
    }

    if XGB_AVAILABLE:
        models["XGBoost"] = XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, tree_method="hist",
            eval_metric='logloss', random_state=42
        )

    print(f"\n{'Model Architecture':<18} | {'Acc':<5} | {'CV 95% CI':<13} | {'AUC':<5} | {'Size(KB)':<8} | {'Train(s)':<8} | {'Lat(μs)':<8}")
    print("-" * 90)

    benchmark_results = []
    metadata = {}

    # ==========================================
    # 5. EXECUTION GAUNTLET
    # ==========================================
    for name, model in models.items():
        # Inner CV (5-Fold) using Robust Balancer
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = []

        for train_idx, val_idx in skf.split(X_train_outer, y_train_outer):
            X_fold_train, y_fold_train = X_train_outer.iloc[train_idx], y_train_outer.iloc[train_idx]
            X_fold_val, y_fold_val = X_train_outer.iloc[val_idx], y_train_outer.iloc[val_idx]

            X_fold_bal, y_fold_bal = balance_data(X_fold_train, y_fold_train)

            model.fit(X_fold_bal, y_fold_bal)
            preds = model.predict(X_fold_val.values)
            cv_scores.append(f1_score(y_fold_val, preds, zero_division=0))

        # Calculate 95% Confidence Interval for the F1 Score
        stderr = np.std(cv_scores, ddof=1) / np.sqrt(len(cv_scores))
        ci95 = 1.96 * stderr
        f1_mean = np.mean(cv_scores)
        ci_string = f"[{f1_mean-ci95:.3f},{f1_mean+ci95:.3f}]"

        # Final Training on fully balanced Outer Set
        start_train = time.perf_counter()
        model.fit(X_train_bal, y_train_bal)
        train_time = time.perf_counter() - start_train

        preds = model.predict(X_test_np)
        probs = model.predict_proba(X_test_np)[:, 1] if hasattr(model, "predict_proba") else [0]*len(preds)

        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec = recall_score(y_test, preds, zero_division=0)
        roc = roc_auc_score(y_test, probs) if hasattr(model, "predict_proba") else 0.0
        cm = confusion_matrix(y_test, preds)

        buffer = io.BytesIO()
        joblib.dump(model, buffer)
        size_kb = len(buffer.getvalue()) / 1024.0

        for _ in range(10): model.predict(test_batch)
        start_lat = time.perf_counter_ns()
        for _ in range(100): model.predict(test_batch)
        end_lat = time.perf_counter_ns()

        num_packets = len(test_batch) * 100
        avg_latency_us = ((end_lat - start_lat) / num_packets) / 1000.0

        print(f"{name:<18} | {acc:<5.3f} | {ci_string:<13} | {roc:<5.3f} | {size_kb:<8.2f} | {train_time:<8.3f} | {avg_latency_us:<8.2f}")

        benchmark_results.append({
            "Model": name, "Accuracy": acc, "F1_Mean": f1_mean, "Precision": prec,
            "Recall": rec, "ROC-AUC": roc, "TrainTime_s": train_time,
            "Latency_us": avg_latency_us, "Size_KB": size_kb
        })

        if name == "V7.1 Edge Shield":
            metadata = {
                "model": model, "features": features, "version": "V7.1",
                "training_date": str(datetime.datetime.now()),
                "dataset": "UNSW_2018_IoT_Botnet_Full",
                "class_distribution": {"Safe": int(len(y_train_outer[y_train_outer==0])), "Attack": int(len(y_train_outer[y_train_outer==1]))}
            }

    # ==========================================
    # 6. EXPORT ARTIFACTS
    # ==========================================
    pd.DataFrame(benchmark_results).to_csv("v7_1_benchmark_results.csv", index=False)
    joblib.dump(metadata, 'v7_1_master_shield.pkl')
    print("\nBenchmark Complete. Artifacts saved: 'v7_1_benchmark_results.csv' & 'v7_1_master_shield.pkl'.")

if __name__ == "__main__":
    run_v7_1_academic_benchmark()

Initializing V7.1 Master Academic Edge-Hardware Benchmark...

Loading all dataset chunks. This may take a moment...
Successfully stitched 4 files. Total raw packets: 3,668,522

--- Traffic Distribution (Outer Split) ---
Training Set : Safe (382) | Attack (2,934,435)
Holdout Set  : Safe (95) | Attack (733,610)

Balancing training data to ensure fair evaluation...

Model Architecture | Acc   | CV 95% CI     | AUC   | Size(KB) | Train(s) | Lat(μs) 
------------------------------------------------------------------------------------------
DNN (MLP)          | 0.936 | [0.958,0.964] | 0.976 | 86.66    | 0.312    | 6.11    
Random Forest      | 0.955 | [0.972,0.979] | 0.996 | 234.82   | 0.216    | 107.34  
V7.1 Edge Shield   | 0.951 | [0.964,0.974] | 0.981 | 3.51     | 0.006    | 1.69    
XGBoost            | 0.959 | [0.974,0.980] | 0.996 | 129.22   | 0.046    | 39.23   

Benchmark Complete. Artifacts saved: 'v7_1_benchmark_results.csv' & 'v7_1_master_shield.pkl'.


In [7]:
import joblib
import time
import numpy as np
import sys

# ANSI Color Codes for Cinematic Terminal Output
GREEN = '\033[92m'
RED = '\033[91m'
YELLOW = '\033[93m'
CYAN = '\033[96m'
RESET = '\033[0m'

def run_live_sentinel():
    print(f"{CYAN}==================================================")
    print("V7.1 EDGE SHIELD : LIVE NETWORK SENTINEL")
    print(f"=================================================={RESET}\n")

    # 1. Boot up the Intelligence Matrix
    print(f"{YELLOW}[SYSTEM] Booting V7.1 Master Shield...{RESET}")
    try:
        metadata = joblib.load('v7_1_master_shield.pkl')
        shield_engine = metadata['model']
        print(f"{GREEN}[SUCCESS] Neural Tree loaded ({metadata['training_date']}). Shield Active.{RESET}\n")
    except FileNotFoundError:
        print(f"{RED}[FATAL] v7_1_master_shield.pkl not found. Run the benchmark script first.{RESET}")
        return

    print("Listening for inbound traffic on interface eth0...\n")
    time.sleep(2)

    # 2. Simulation Variables
    packets_processed = 0
    threats_blocked = 0
    fast_lane_skips = 0
    start_time = time.time()

    try:
        # Phase 1: Normal Home Network Traffic (Smart bulbs, Netflix, Web browsing)
        print(f"{CYAN}--- PHASE 1: STANDARD HOME TRAFFIC ---{RESET}")
        for _ in range(15):
            # Normal Profile: Moderate bytes, high variance in timing
            mean_time = np.random.uniform(0.1, 1.5)
            stddev = np.random.uniform(0.05, 0.5)
            bytes_vol = np.random.randint(64, 1500)
            seq = np.random.randint(1, 1000)

            packet = np.array([[mean_time, stddev, bytes_vol, seq]])

            # Predict
            is_attack = shield_engine.predict(packet)[0]

            if is_attack == 0:
                print(f"{GREEN}[ALLOW] IP: 192.168.1.{np.random.randint(2, 50):03d} | Size: {bytes_vol:4d}B | Status: BENIGN{RESET}")

            packets_processed += 1
            time.sleep(0.3) # Slow human-like pacing

        # Phase 2: The Mirai Volumetric Attack
        print(f"\n{RED}!!! WARNING: VOLUMETRIC ANOMALY DETECTED !!!{RESET}")
        print(f"{CYAN}--- PHASE 2: INBOUND MIRAI BOTNET FLOOD ---{RESET}")
        time.sleep(1)

        # The hacker blasts the router with thousands of packets in a fraction of a second
        for i in range(1, 101):
            # Mirai Profile: Tiny payload, perfectly uniform automated speed (no jitter)
            mean_time = np.random.uniform(0.0001, 0.001)
            stddev = np.random.uniform(0.0000, 0.0001)
            bytes_vol = np.random.randint(40, 64)
            seq = i

            packet = np.array([[mean_time, stddev, bytes_vol, seq]])

            # THE FAST-LANE PROBABILITY ALGORITHM (1-in-10 inspection for the demo)
            if np.random.rand() > 0.1:
                fast_lane_skips += 1
                # Silently block based on established flow rules (saves CPU)
                sys.stdout.write(f"{YELLOW}x{RESET}")
                sys.stdout.flush()
                threats_blocked += 1
            else:
                # Deep Inspection by the AI
                is_attack = shield_engine.predict(packet)[0]
                if is_attack == 1:
                    print(f"\n{RED}[BLOCK] MALICIOUS SIGNATURE DETECTED | Sequence {seq:04d} | Shielding CPU...{RESET}")
                    threats_blocked += 1

            packets_processed += 1
            time.sleep(0.02) # Blistering fast machine pacing

        # 3. After Action Report
        total_time = time.time() - start_time
        print(f"\n\n{CYAN}==================================================")
        print("ATTACK NEUTRALIZED : AFTER ACTION REPORT")
        print(f"=================================================={RESET}")
        print(f"Total Packets Analyzed : {packets_processed}")
        print(f"Threats Blocked        : {threats_blocked}")
        print(f"Fast-Lane CPU Skips    : {fast_lane_skips} packets offloaded from CPU")
        print(f"System Uptime Intact   : True")
        print(f"{CYAN}=================================================={RESET}\n")

    except KeyboardInterrupt:
        print(f"\n{YELLOW}[SYSTEM] Manual override engaged. Shield shutting down.{RESET}")

if __name__ == "__main__":
    run_live_sentinel()

V7.1 EDGE SHIELD : LIVE NETWORK SENTINEL

[SYSTEM] Booting V7.1 Master Shield...
[SUCCESS] Neural Tree loaded (2026-07-02 08:39:57.395625). Shield Active.

Listening for inbound traffic on interface eth0...

--- PHASE 1: STANDARD HOME TRAFFIC ---
[ALLOW] IP: 192.168.1.022 | Size: 1194B | Status: BENIGN
[ALLOW] IP: 192.168.1.025 | Size:  394B | Status: BENIGN
[ALLOW] IP: 192.168.1.003 | Size:  194B | Status: BENIGN
[ALLOW] IP: 192.168.1.022 | Size:  449B | Status: BENIGN
[ALLOW] IP: 192.168.1.026 | Size:  316B | Status: BENIGN
[ALLOW] IP: 192.168.1.016 | Size:  763B | Status: BENIGN
[ALLOW] IP: 192.168.1.004 | Size:  626B | Status: BENIGN
[ALLOW] IP: 192.168.1.019 | Size:  904B | Status: BENIGN
[ALLOW] IP: 192.168.1.027 | Size:  305B | Status: BENIGN
[ALLOW] IP: 192.168.1.008 | Size: 1454B | Status: BENIGN
[ALLOW] IP: 192.168.1.018 | Size:   98B | Status: BENIGN
[ALLOW] IP: 192.168.1.043 | Size: 1089B | Status: BENIGN
[ALLOW] IP: 192.168.1.045 | Size:  465B | Status: BENIGN
[ALLOW] IP: 

In [8]:
import joblib
import time
import numpy as np
import sys
import csv
from datetime import datetime

# ANSI Color Codes
GREEN = '\033[92m'
RED = '\033[91m'
YELLOW = '\033[93m'
CYAN = '\033[96m'
RESET = '\033[0m'

CACHE_TTL = 30.0 # Flow cache expiration in seconds

# ==========================================
# 1. CORE LOGIC & INITIALIZATION
# ==========================================
def load_model_and_metadata(filepath):
    try:
        metadata = joblib.load(filepath)
        print(f"{CYAN}==================================================")
        print("V9 EDGE SHIELD : SCIENTIFIC IDS PROTOTYPE")
        print(f"=================================================={RESET}")
        print(f"Architecture    : Decision Tree (Depth 5)")
        print(f"Dataset         : {metadata.get('dataset', 'UNSW_2018_IoT_Botnet')}")
        print(f"Training Date   : {metadata.get('training_date', 'Unknown')}")
        print(f"{CYAN}=================================================={RESET}\n")
        return metadata['model']
    except FileNotFoundError:
        print(f"{RED}[FATAL ERROR] {filepath} not found.{RESET}")
        sys.exit(1)

def inspect_packet(model, packet_features, flow_key, flow_cache):
    """
    Simulates a stateful firewall's 5-Tuple flow cache with TTL expiration.
    """
    is_cached = False
    current_time = time.time()
    start_ns = time.perf_counter_ns()

    # Check cache and TTL
    if flow_key in flow_cache and (current_time - flow_cache[flow_key]['timestamp'] <= CACHE_TTL):
        prediction = flow_cache[flow_key]['pred']
        probability = flow_cache[flow_key]['prob']
        is_cached = True
    else:
        # Deep Inference
        prediction = model.predict(packet_features)[0]
        probability = model.predict_proba(packet_features)[0][prediction]
        # Update Cache
        flow_cache[flow_key] = {
            'pred': prediction,
            'prob': probability,
            'timestamp': current_time
        }

    end_ns = time.perf_counter_ns()
    latency_us = (end_ns - start_ns) / 1000.0

    return prediction, probability, is_cached, latency_us

# ==========================================
# 2. 5-TUPLE TRAFFIC GENERATORS
# ==========================================
def generate_benign_profile():
    profiles = [
        {"name": "Smart TV Streaming", "mean": np.random.uniform(0.01, 0.05), "stddev": np.random.uniform(0.001, 0.01), "bytes": np.random.randint(1200, 1500), "proto": "UDP", "dport": 443},
        {"name": "Smart Bulb Keepalive", "mean": np.random.uniform(1.0, 5.0), "stddev": np.random.uniform(0.1, 0.5), "bytes": np.random.randint(40, 80), "proto": "TCP", "dport": 1883},
        {"name": "Laptop Web Browsing", "mean": np.random.uniform(0.1, 0.8), "stddev": np.random.uniform(0.05, 0.2), "bytes": np.random.randint(200, 800), "proto": "TCP", "dport": 80}
    ]
    p = np.random.choice(profiles)
    src_ip = f"192.168.1.{np.random.randint(10, 50)}"
    dst_ip = "192.168.1.1" # Router
    src_port = np.random.randint(1024, 65535)

    flow_key = (src_ip, dst_ip, src_port, p['dport'], p['proto'])
    features = np.array([[p['mean'], p['stddev'], p['bytes'], np.random.randint(1, 1000)]])
    return flow_key, p['name'], features

def generate_malicious_profile(seq, persistent_ip=None, persistent_port=None):
    profiles = [
        {"name": "Mirai SYN Flood", "mean": np.random.uniform(0.0001, 0.0005), "stddev": 0.0, "bytes": np.random.randint(40, 64), "proto": "TCP"},
        {"name": "UDP Volumetric", "mean": np.random.uniform(0.0005, 0.001), "stddev": np.random.uniform(0.0, 0.0001), "bytes": np.random.randint(512, 1024), "proto": "UDP"},
        {"name": "ICMP Ping Sweep", "mean": np.random.uniform(0.01, 0.05), "stddev": np.random.uniform(0.001, 0.01), "bytes": 84, "proto": "ICMP"},
        {"name": "TCP Port Scan", "mean": np.random.uniform(0.001, 0.01), "stddev": np.random.uniform(0.0, 0.001), "bytes": 40, "proto": "TCP"}
    ]
    p = np.random.choice(profiles)

    src_ip = persistent_ip or f"{np.random.choice(['185', '91', '103'])}.{np.random.randint(1,255)}.{np.random.randint(1,255)}.{np.random.randint(1,255)}"
    dst_ip = "192.168.1.1"
    src_port = persistent_port or np.random.randint(1024, 65535)

    # Port scans hit rotating destination ports
    dst_port = np.random.randint(1, 1024) if p['name'] == "TCP Port Scan" else 80

    flow_key = (src_ip, dst_ip, src_port, dst_port, p['proto'])
    features = np.array([[p['mean'], p['stddev'], p['bytes'], seq]])
    return flow_key, p['name'], features

# ==========================================
# 3. MASTER SIMULATION LOOP
# ==========================================
def run_simulation():
    np.random.seed(42) # Reproducible demonstrations
    model = load_model_and_metadata('v7_1_master_shield.pkl')

    flow_cache = {}
    log_file = open('sentinel_log.csv', 'w', newline='')
    log_writer = csv.writer(log_file)
    log_writer.writerow(['Timestamp', 'FlowKey', 'Profile', 'Prediction', 'Confidence', 'Cached', 'Latency_us'])

    stats = {
        'total': 0, 'blocked': 0, 'false_pos': 0, 'ai_inferences': 0, 'cached_skips': 0,
        'ai_latencies': [], 'cache_latencies': [], 'total_algorithmic_time_ns': 0
    }

    print(f"{GREEN}[SYSTEM] Inference Engine Ready. Listening on interface eth0...{RESET}\n")
    time.sleep(1.5)

    try:
        # --- PHASE 1: BENIGN TRAFFIC ---
        print(f"{CYAN}--- PHASE 1: STANDARD HOME TRAFFIC ---{RESET}")
        for _ in range(15):
            flow_key, device, features = generate_benign_profile()
            pred, prob, is_cached, lat = inspect_packet(model, features, flow_key, flow_cache)

            # Record algorithmic time minus simulation sleeps
            stats['total_algorithmic_time_ns'] += (lat * 1000)
            stats['total'] += 1
            if is_cached:
                stats['cached_skips'] += 1
                stats['cache_latencies'].append(lat)
            else:
                stats['ai_inferences'] += 1
                stats['ai_latencies'].append(lat)

            # Format Flow Key for display: IP:Port
            flow_str = f"{flow_key[0]}:{flow_key[2]}"

            log_writer.writerow([datetime.now().isoformat(), str(flow_key), device, pred, f"{prob:.4f}", is_cached, f"{lat:.2f}"])

            if pred == 0:
                print(f"{GREEN}[ALLOW] Flow: {flow_str:<21} | {device:<20} | Conf: {prob*100:5.1f}% | Lat: {lat:5.1f} μs{RESET}")
            else:
                print(f"{YELLOW}[WARNING] False Positive Detected on Flow: {flow_str}{RESET}")
                stats['false_pos'] += 1

            time.sleep(0.4)

        # --- PHASE 2: ATTACK TRAFFIC ---
        print(f"\n{RED}!!! ANOMALY DETECTED: HIGH-VOLUME CONNECTION ATTEMPTS !!!{RESET}")
        print(f"{CYAN}--- PHASE 2: INBOUND BOTNET FLOOD ---{RESET}")
        time.sleep(1)

        # 3 persistent botnet nodes orchestrating the flood
        botnet_nodes = [(f"185.22.44.{i}", np.random.randint(10000, 60000)) for i in range(1, 4)]

        for i in range(1, 101):
            b_ip, b_port = botnet_nodes[i % 3] # Rotate through the 3 botnet nodes
            flow_key, attack_type, features = generate_malicious_profile(seq=i, persistent_ip=b_ip, persistent_port=b_port)

            pred, prob, is_cached, lat = inspect_packet(model, features, flow_key, flow_cache)

            stats['total_algorithmic_time_ns'] += (lat * 1000)
            stats['total'] += 1

            if is_cached:
                stats['cached_skips'] += 1
                stats['cache_latencies'].append(lat)
            else:
                stats['ai_inferences'] += 1
                stats['ai_latencies'].append(lat)

            log_writer.writerow([datetime.now().isoformat(), str(flow_key), attack_type, pred, f"{prob:.4f}", is_cached, f"{lat:.2f}"])

            if pred == 1:
                stats['blocked'] += 1
                if is_cached:
                    sys.stdout.write(f"{YELLOW}x{RESET}")
                    sys.stdout.flush()
                else:
                    print(f"\n{RED}[BLOCK] {attack_type:<16} | Src: {flow_key[0]:<15} | Conf: {prob*100:5.1f}% | Engine: AI{RESET}")
            else:
                print(f"\n{YELLOW}[WARNING] False Negative - Attack Packet Evaded Detection!{RESET}")

            time.sleep(0.02)

        # --- PHASE 3: DIAGNOSTIC REPORT ---
        algo_time_sec = stats['total_algorithmic_time_ns'] / 1e9
        pps = stats['total'] / algo_time_sec if algo_time_sec > 0 else 0
        reduction = (stats['cached_skips'] / stats['total']) * 100

        avg_cache_lat = np.mean(stats['cache_latencies']) if stats['cache_latencies'] else 0
        avg_ai_lat = np.mean(stats['ai_latencies']) if stats['ai_latencies'] else 0

        print(f"\n\n{CYAN}==================================================")
        print("SIMULATION COMPLETE : SCIENTIFIC METRICS")
        print(f"=================================================={RESET}")
        print(f"Total Packets Processed : {stats['total']}")
        print(f"Actual Threats Blocked  : {stats['blocked']}")
        print(f"False Positives         : {stats['false_pos']}")
        print(f"Algorithmic Throughput  : {pps:,.0f} packets/sec")
        print(f"--------------------------------------------------")
        print(f"AI Deep Inferences      : {stats['ai_inferences']} (Avg Latency: {avg_ai_lat:.1f} μs)")
        print(f"Flow Cache Skips        : {stats['cached_skips']} (Avg Latency: {avg_cache_lat:.1f} μs)")
        print(f"Inference Reduction     : {GREEN}{reduction:.1f}% AI Workload Bypassed{RESET}")
        print(f"{CYAN}=================================================={RESET}")
        print(f"Detailed 5-tuple logs exported to: sentinel_log.csv")

    except KeyboardInterrupt:
        print(f"\n{YELLOW}[SYSTEM] Manual termination sequence initiated.{RESET}")
    finally:
        log_file.close()

if __name__ == "__main__":
    run_simulation()

V9 EDGE SHIELD : SCIENTIFIC IDS PROTOTYPE
Architecture    : Decision Tree (Depth 5)
Dataset         : UNSW_2018_IoT_Botnet_Full
Training Date   : 2026-07-02 08:39:57.395625

[SYSTEM] Inference Engine Ready. Listening on interface eth0...

--- PHASE 1: STANDARD HOME TRAFFIC ---
[ALLOW] Flow: 192.168.1.31:1793     | Laptop Web Browsing  | Conf:  90.0% | Lat: 4126.1 μs
[ALLOW] Flow: 192.168.1.37:3771     | Laptop Web Browsing  | Conf:  90.0% | Lat: 3096.3 μs
[WARNING] False Positive Detected on Flow: 192.168.1.13:9816
[ALLOW] Flow: 192.168.1.26:27555    | Smart Bulb Keepalive | Conf: 100.0% | Lat: 3804.2 μs
[ALLOW] Flow: 192.168.1.24:30151    | Laptop Web Browsing  | Conf:  90.0% | Lat: 997.6 μs
[ALLOW] Flow: 192.168.1.38:52238    | Smart TV Streaming   | Conf:  98.5% | Lat: 4276.5 μs
[ALLOW] Flow: 192.168.1.14:4585     | Smart TV Streaming   | Conf:  98.5% | Lat: 4585.6 μs
[ALLOW] Flow: 192.168.1.44:48740    | Smart TV Streaming   | Conf:  98.5% | Lat: 1010.2 μs
[ALLOW] Flow: 192.168.1.4

In [9]:
import joblib
import time
import numpy as np
import pandas as pd
import sys
import csv
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ANSI Color Codes
GREEN = '\033[92m'
RED = '\033[91m'
YELLOW = '\033[93m'
CYAN = '\033[96m'
RESET = '\033[0m'

CACHE_TTL = 30.0

def load_model_and_metadata(filepath):
    try:
        metadata = joblib.load(filepath)
        print(f"{CYAN}==================================================")
        print("V9.1 EDGE SHIELD : EMPIRICAL TRAFFIC REPLAYER")
        print(f"=================================================={RESET}")
        print(f"Architecture    : Decision Tree (Depth 5)")
        print(f"Training Date   : {metadata.get('training_date', 'Unknown')}")
        print(f"{CYAN}=================================================={RESET}\n")
        return metadata['model'], metadata.get('features', ['mean', 'stddev', 'bytes', 'seq'])
    except FileNotFoundError:
        print(f"{RED}[FATAL ERROR] {filepath} not found.{RESET}")
        sys.exit(1)

# ==========================================
# EMPIRICAL DATA LOADER (The Fix)
# ==========================================
def load_empirical_traffic(features):
    print(f"{YELLOW}[SYSTEM] Loading empirical traffic from CSV for live replay...{RESET}")
    try:
        df = pd.read_csv('UNSW_2018_IoT_Botnet_Full5pc_1.csv', low_memory=False)
        df = df.dropna(subset=features + ['attack'])
        for col in features:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df['attack'] = pd.to_numeric(df['attack'], errors='coerce').astype(int)
        df = df.dropna(subset=features + ['attack'])

        benign_pool = df[df['attack'] == 0][features].values
        attack_pool = df[df['attack'] == 1][features].values
        print(f"{GREEN}[SUCCESS] Loaded {len(benign_pool)} benign and {len(attack_pool)} attack packets for replay.{RESET}\n")
        return benign_pool, attack_pool
    except FileNotFoundError:
        print(f"{RED}[FATAL ERROR] UNSW CSV not found. Replayer requires real data.{RESET}")
        sys.exit(1)

def inspect_packet(model, packet_features, flow_key, flow_cache):
    is_cached = False
    current_time = time.time()
    start_ns = time.perf_counter_ns()

    if flow_key in flow_cache and (current_time - flow_cache[flow_key]['timestamp'] <= CACHE_TTL):
        prediction = flow_cache[flow_key]['pred']
        probability = flow_cache[flow_key]['prob']
        is_cached = True
    else:
        prediction = model.predict(packet_features)[0]
        probability = model.predict_proba(packet_features)[0][prediction]
        flow_cache[flow_key] = {'pred': prediction, 'prob': probability, 'timestamp': current_time}

    end_ns = time.perf_counter_ns()
    latency_us = (end_ns - start_ns) / 1000.0
    return prediction, probability, is_cached, latency_us

def run_simulation():
    np.random.seed(42)
    model, features = load_model_and_metadata('v7_1_master_shield.pkl')
    benign_pool, attack_pool = load_empirical_traffic(features)

    flow_cache = {}
    log_file = open('sentinel_log.csv', 'w', newline='')
    log_writer = csv.writer(log_file)
    log_writer.writerow(['Timestamp', 'FlowKey', 'Profile', 'Prediction', 'Confidence', 'Cached', 'Latency_us'])

    stats = {'total': 0, 'blocked': 0, 'false_pos': 0, 'ai_inferences': 0, 'cached_skips': 0, 'ai_latencies': [], 'cache_latencies': []}

    time.sleep(1.5)

    try:
        # --- PHASE 1: BENIGN TRAFFIC ---
        print(f"{CYAN}--- PHASE 1: STANDARD HOME TRAFFIC (REPLAY) ---{RESET}")
        for _ in range(15):
            # Pull a real benign packet from the CSV
            packet_features = benign_pool[np.random.randint(0, len(benign_pool))].reshape(1, -1)

            flow_key = (f"192.168.1.{np.random.randint(10, 50)}", "192.168.1.1", np.random.randint(1024, 65535), 443, "TCP")
            pred, prob, is_cached, lat = inspect_packet(model, packet_features, flow_key, flow_cache)

            stats['total'] += 1
            if is_cached:
                stats['cached_skips'] += 1
                stats['cache_latencies'].append(lat)
            else:
                stats['ai_inferences'] += 1
                stats['ai_latencies'].append(lat)

            if pred == 0:
                print(f"{GREEN}[ALLOW] Flow: {flow_key[0]}:{flow_key[2]:<5} | Status: BENIGN | Conf: {prob*100:5.1f}% | Lat: {lat:6.1f} μs{RESET}")
            else:
                print(f"{YELLOW}[WARNING] False Positive Detected{RESET}")
                stats['false_pos'] += 1

            time.sleep(0.4)

        # --- PHASE 2: ATTACK TRAFFIC ---
        print(f"\n{RED}!!! ANOMALY DETECTED: BOTNET FLOOD !!!{RESET}")
        print(f"{CYAN}--- PHASE 2: EMPIRICAL ATTACK REPLAY ---{RESET}")
        time.sleep(1)

        botnet_nodes = [(f"185.22.44.{i}", np.random.randint(10000, 60000)) for i in range(1, 4)]

        for i in range(1, 101):
            b_ip, b_port = botnet_nodes[i % 3]

            # Pull a REAL malicious packet directly from the CSV
            packet_features = attack_pool[np.random.randint(0, len(attack_pool))].reshape(1, -1)
            flow_key = (b_ip, "192.168.1.1", b_port, 80, "TCP")

            pred, prob, is_cached, lat = inspect_packet(model, packet_features, flow_key, flow_cache)

            stats['total'] += 1
            if is_cached:
                stats['cached_skips'] += 1
                stats['cache_latencies'].append(lat)
            else:
                stats['ai_inferences'] += 1
                stats['ai_latencies'].append(lat)

            if pred == 1:
                stats['blocked'] += 1
                if is_cached:
                    sys.stdout.write(f"{YELLOW}x{RESET}")
                    sys.stdout.flush()
                else:
                    print(f"\n{RED}[BLOCK] EMPIRICAL BOTNET SIGNATURE | Src: {b_ip:<15} | Conf: {prob*100:5.1f}%{RESET}")
            else:
                print(f"\n{YELLOW}[WARNING] False Negative!{RESET}")

            time.sleep(0.02)

        # --- PHASE 3: DIAGNOSTIC REPORT ---
        reduction = (stats['cached_skips'] / stats['total']) * 100
        avg_cache_lat = np.mean(stats['cache_latencies']) if stats['cache_latencies'] else 0
        avg_ai_lat = np.mean(stats['ai_latencies']) if stats['ai_latencies'] else 0

        print(f"\n\n{CYAN}==================================================")
        print("SIMULATION COMPLETE : EMPIRICAL METRICS")
        print(f"=================================================={RESET}")
        print(f"Total Packets Processed : {stats['total']}")
        print(f"Actual Threats Blocked  : {stats['blocked']} / 100")
        print(f"False Positives         : {stats['false_pos']} / 15")
        print(f"--------------------------------------------------")
        print(f"AI Single-Row Overhead  : {avg_ai_lat:.1f} μs (Python limitation)")
        print(f"Flow Cache Latency      : {avg_cache_lat:.1f} μs (Hardware speed)")
        print(f"Inference Reduction     : {GREEN}{reduction:.1f}% AI Workload Bypassed{RESET}")
        print(f"{CYAN}=================================================={RESET}")

    except KeyboardInterrupt:
        pass

if __name__ == "__main__":
    run_simulation()

V9.1 EDGE SHIELD : EMPIRICAL TRAFFIC REPLAYER
Architecture    : Decision Tree (Depth 5)
Training Date   : 2026-07-02 08:39:57.395625

[SYSTEM] Loading empirical traffic from CSV for live replay...
[SUCCESS] Loaded 0 benign and 1000000 attack packets for replay.

--- PHASE 1: STANDARD HOME TRAFFIC (REPLAY) ---


ValueError: high <= 0

In [10]:
import joblib
import time
import os
import numpy as np
import pandas as pd
import sys
import csv
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ANSI Color Codes
GREEN = '\033[92m'
RED = '\033[91m'
YELLOW = '\033[93m'
CYAN = '\033[96m'
RESET = '\033[0m'

CACHE_TTL = 30.0

def load_model_and_metadata(filepath):
    try:
        metadata = joblib.load(filepath)
        print(f"{CYAN}==================================================")
        print("V9.1 EDGE SHIELD : EMPIRICAL TRAFFIC REPLAYER")
        print(f"=================================================={RESET}")
        print(f"Architecture    : Decision Tree (Depth 5)")
        print(f"Training Date   : {metadata.get('training_date', 'Unknown')}")
        print(f"{CYAN}=================================================={RESET}\n")
        return metadata['model'], metadata.get('features', ['mean', 'stddev', 'bytes', 'seq'])
    except FileNotFoundError:
        print(f"{RED}[FATAL ERROR] {filepath} not found.{RESET}")
        sys.exit(1)

# ==========================================
# EMPIRICAL DATA LOADER (Upgraded to load all files)
# ==========================================
def load_empirical_traffic(features):
    print(f"{YELLOW}[SYSTEM] Loading empirical traffic from all CSV chunks...{RESET}")
    file_names = [
        'UNSW_2018_IoT_Botnet_Full5pc_1.csv',
        'UNSW_2018_IoT_Botnet_Full5pc_2.csv',
        'UNSW_2018_IoT_Botnet_Full5pc_3.csv',
        'UNSW_2018_IoT_Botnet_Full5pc_4.csv'
    ]

    dataframes = [pd.read_csv(f, low_memory=False) for f in file_names if os.path.exists(f)]

    if not dataframes:
        print(f"{RED}[FATAL ERROR] No CSVs found in this directory.{RESET}")
        sys.exit(1)

    df = pd.concat(dataframes, ignore_index=True)

    for col in features:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['attack'] = pd.to_numeric(df['attack'], errors='coerce')

    # Drop corrupted rows
    df = df.dropna(subset=features + ['attack'])
    df['attack'] = df['attack'].astype(int)

    benign_pool = df[df['attack'] == 0][features].values
    attack_pool = df[df['attack'] == 1][features].values

    print(f"{GREEN}[SUCCESS] Loaded {len(benign_pool)} benign and {len(attack_pool)} attack packets for replay.{RESET}\n")
    return benign_pool, attack_pool

def inspect_packet(model, packet_features, flow_key, flow_cache):
    is_cached = False
    current_time = time.time()
    start_ns = time.perf_counter_ns()

    if flow_key in flow_cache and (current_time - flow_cache[flow_key]['timestamp'] <= CACHE_TTL):
        prediction = flow_cache[flow_key]['pred']
        probability = flow_cache[flow_key]['prob']
        is_cached = True
    else:
        prediction = model.predict(packet_features)[0]
        probability = model.predict_proba(packet_features)[0][prediction]
        flow_cache[flow_key] = {'pred': prediction, 'prob': probability, 'timestamp': current_time}

    end_ns = time.perf_counter_ns()
    latency_us = (end_ns - start_ns) / 1000.0
    return prediction, probability, is_cached, latency_us

def run_simulation():
    np.random.seed(42)
    model, features = load_model_and_metadata('v7_1_master_shield.pkl')
    benign_pool, attack_pool = load_empirical_traffic(features)

    flow_cache = {}
    log_file = open('sentinel_log.csv', 'w', newline='')
    log_writer = csv.writer(log_file)
    log_writer.writerow(['Timestamp', 'FlowKey', 'Profile', 'Prediction', 'Confidence', 'Cached', 'Latency_us'])

    stats = {'total': 0, 'blocked': 0, 'false_pos': 0, 'ai_inferences': 0, 'cached_skips': 0, 'ai_latencies': [], 'cache_latencies': []}
    time.sleep(1.5)

    try:
        # --- PHASE 1: BENIGN TRAFFIC ---
        print(f"{CYAN}--- PHASE 1: STANDARD HOME TRAFFIC ---{RESET}")
        for _ in range(15):
            # Smart Fallback: Use real benign packets if available, otherwise use strict synthetic baseline
            if len(benign_pool) > 0:
                packet_features = benign_pool[np.random.randint(0, len(benign_pool))].reshape(1, -1)
                profile_name = "Empirical Benign"
            else:
                packet_features = np.array([[np.random.uniform(0.1, 0.8), np.random.uniform(0.05, 0.2), np.random.randint(200, 800), np.random.randint(1, 1000)]])
                profile_name = "Baseline Simulation"

            flow_key = (f"192.168.1.{np.random.randint(10, 50)}", "192.168.1.1", np.random.randint(1024, 65535), 443, "TCP")
            pred, prob, is_cached, lat = inspect_packet(model, packet_features, flow_key, flow_cache)

            stats['total'] += 1
            if is_cached:
                stats['cached_skips'] += 1
                stats['cache_latencies'].append(lat)
            else:
                stats['ai_inferences'] += 1
                stats['ai_latencies'].append(lat)

            if pred == 0:
                print(f"{GREEN}[ALLOW] Flow: {flow_key[0]}:{flow_key[2]:<5} | Status: BENIGN | Conf: {prob*100:5.1f}% | Lat: {lat:6.1f} μs{RESET}")
            else:
                print(f"{YELLOW}[WARNING] False Positive Detected{RESET}")
                stats['false_pos'] += 1

            time.sleep(0.4)

        # --- PHASE 2: ATTACK TRAFFIC ---
        print(f"\n{RED}!!! ANOMALY DETECTED: BOTNET FLOOD !!!{RESET}")
        print(f"{CYAN}--- PHASE 2: EMPIRICAL ATTACK REPLAY ---{RESET}")
        time.sleep(1)

        botnet_nodes = [(f"185.22.44.{i}", np.random.randint(10000, 60000)) for i in range(1, 4)]

        for i in range(1, 101):
            b_ip, b_port = botnet_nodes[i % 3]

            # Pull a REAL malicious packet directly from the massive 3-million packet CSV pool
            packet_features = attack_pool[np.random.randint(0, len(attack_pool))].reshape(1, -1)
            flow_key = (b_ip, "192.168.1.1", b_port, 80, "TCP")

            pred, prob, is_cached, lat = inspect_packet(model, packet_features, flow_key, flow_cache)

            stats['total'] += 1
            if is_cached:
                stats['cached_skips'] += 1
                stats['cache_latencies'].append(lat)
            else:
                stats['ai_inferences'] += 1
                stats['ai_latencies'].append(lat)

            if pred == 1:
                stats['blocked'] += 1
                if is_cached:
                    sys.stdout.write(f"{YELLOW}x{RESET}")
                    sys.stdout.flush()
                else:
                    print(f"\n{RED}[BLOCK] EMPIRICAL BOTNET SIGNATURE | Src: {b_ip:<15} | Conf: {prob*100:5.1f}%{RESET}")
            else:
                print(f"\n{YELLOW}[WARNING] False Negative!{RESET}")

            time.sleep(0.02)

        # --- PHASE 3: DIAGNOSTIC REPORT ---
        reduction = (stats['cached_skips'] / stats['total']) * 100
        avg_cache_lat = np.mean(stats['cache_latencies']) if stats['cache_latencies'] else 0
        avg_ai_lat = np.mean(stats['ai_latencies']) if stats['ai_latencies'] else 0

        print(f"\n\n{CYAN}==================================================")
        print("SIMULATION COMPLETE : EMPIRICAL METRICS")
        print(f"=================================================={RESET}")
        print(f"Total Packets Processed : {stats['total']}")
        print(f"Actual Threats Blocked  : {stats['blocked']} / 100")
        print(f"False Positives         : {stats['false_pos']} / 15")
        print(f"--------------------------------------------------")
        print(f"AI Deep Inferences      : {stats['ai_inferences']} (Avg Lat: {avg_ai_lat:.1f} μs)")
        print(f"Flow Cache Skips        : {stats['cached_skips']} (Avg Lat: {avg_cache_lat:.1f} μs)")
        print(f"Inference Reduction     : {GREEN}{reduction:.1f}% AI Workload Bypassed{RESET}")
        print(f"{CYAN}=================================================={RESET}")

    except KeyboardInterrupt:
        pass

if __name__ == "__main__":
    run_simulation()

V9.1 EDGE SHIELD : EMPIRICAL TRAFFIC REPLAYER
Architecture    : Decision Tree (Depth 5)
Training Date   : 2026-07-02 08:39:57.395625

[SYSTEM] Loading empirical traffic from all CSV chunks...
[SUCCESS] Loaded 477 benign and 3668045 attack packets for replay.

--- PHASE 1: STANDARD HOME TRAFFIC ---
[ALLOW] Flow: 192.168.1.38:39182 | Status: BENIGN | Conf:  66.7% | Lat: 1035.6 μs
[ALLOW] Flow: 192.168.1.17:45756 | Status: BENIGN | Conf:  98.5% | Lat:  769.8 μs
[ALLOW] Flow: 192.168.1.48:7289  | Status: BENIGN | Conf:  98.5% | Lat:  836.9 μs
[ALLOW] Flow: 192.168.1.32:38218 | Status: BENIGN | Conf:  98.5% | Lat: 3806.6 μs
[ALLOW] Flow: 192.168.1.33:61812 | Status: BENIGN | Conf:  98.5% | Lat: 3983.5 μs
[ALLOW] Flow: 192.168.1.49:17047 | Status: BENIGN | Conf: 100.0% | Lat: 4100.3 μs
[ALLOW] Flow: 192.168.1.31:1793  | Status: BENIGN | Conf:  98.5% | Lat: 5014.3 μs
[ALLOW] Flow: 192.168.1.39:57125 | Status: BENIGN | Conf:  66.7% | Lat: 6516.9 μs
[ALLOW] Flow: 192.168.1.30:18592 | Status: BE

In [11]:
import joblib
import time
import os
import numpy as np
import pandas as pd
import sys
import csv
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ANSI Color Codes
GREEN = '\033[92m'
RED = '\033[91m'
YELLOW = '\033[93m'
CYAN = '\033[96m'
RESET = '\033[0m'

CACHE_TTL = 30.0

# ==========================================
# MODULE 1: MODEL LOADER
# ==========================================
def load_model_and_metadata(filepath):
    try:
        metadata = joblib.load(filepath)
        print(f"{CYAN}==================================================")
        print("V10 EDGE SHIELD : EDGE IDS EVALUATION PROTOTYPE")
        print(f"=================================================={RESET}")
        print(f"Architecture    : Decision Tree (Depth 5)")
        print(f"Training Date   : {metadata.get('training_date', 'Unknown')}")
        print(f"Note            : Cache effectiveness is heavily dependent on flow reuse.")
        print(f"{CYAN}=================================================={RESET}\n")
        return metadata['model'], metadata.get('features', ['mean', 'stddev', 'bytes', 'seq'])
    except FileNotFoundError:
        print(f"{RED}[FATAL ERROR] {filepath} not found.{RESET}")
        sys.exit(1)

# ==========================================
# MODULE 2: EMPIRICAL DATA LOADER
# ==========================================
def load_empirical_traffic(features):
    print(f"{YELLOW}[SYSTEM] Loading empirical traffic from all CSV chunks...{RESET}")
    file_names = [
        'UNSW_2018_IoT_Botnet_Full5pc_1.csv',
        'UNSW_2018_IoT_Botnet_Full5pc_2.csv',
        'UNSW_2018_IoT_Botnet_Full5pc_3.csv',
        'UNSW_2018_IoT_Botnet_Full5pc_4.csv'
    ]

    dataframes = [pd.read_csv(f, low_memory=False) for f in file_names if os.path.exists(f)]

    if not dataframes:
        print(f"{RED}[FATAL ERROR] No CSVs found in this directory.{RESET}")
        sys.exit(1)

    df = pd.concat(dataframes, ignore_index=True)
    for col in features:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['attack'] = pd.to_numeric(df['attack'], errors='coerce')

    df = df.dropna(subset=features + ['attack'])
    df['attack'] = df['attack'].astype(int)

    benign_pool = df[df['attack'] == 0][features].values
    attack_pool = df[df['attack'] == 1][features].values

    print(f"{GREEN}[SUCCESS] Loaded {len(benign_pool)} benign and {len(attack_pool)} attack packets for replay.{RESET}\n")
    return benign_pool, attack_pool

# ==========================================
# MODULE 3: THE FLOW CACHE ENGINE
# ==========================================
def inspect_packet(model, packet_features, flow_key, flow_cache):
    is_cached = False
    current_time = time.time()
    start_ns = time.perf_counter_ns()

    if flow_key in flow_cache and (current_time - flow_cache[flow_key]['timestamp'] <= CACHE_TTL):
        prediction = flow_cache[flow_key]['pred']
        probability = flow_cache[flow_key]['prob']
        is_cached = True
    else:
        prediction = model.predict(packet_features)[0]
        probability = model.predict_proba(packet_features)[0][prediction]
        flow_cache[flow_key] = {'pred': prediction, 'prob': probability, 'timestamp': current_time}

    end_ns = time.perf_counter_ns()
    latency_us = (end_ns - start_ns) / 1000.0
    return prediction, probability, is_cached, latency_us

# ==========================================
# MODULE 4: MASTER SIMULATOR
# ==========================================
def run_simulation():
    np.random.seed(42)
    model, features = load_model_and_metadata('v7_1_master_shield.pkl')
    benign_pool, attack_pool = load_empirical_traffic(features)

    flow_cache = {}
    log_file = open('v10_evaluation_log.csv', 'w', newline='')
    log_writer = csv.writer(log_file)
    log_writer.writerow(['Timestamp', 'FlowKey', 'Profile', 'GroundTruth', 'Prediction', 'Confidence', 'Cached', 'Latency_us'])

    # Strict Ground Truth Statistics
    stats = {
        'total': 0, 'cached_skips': 0, 'ai_inferences': 0,
        'ai_latencies': [], 'cache_latencies': [], 'algo_time_ns': 0,
        'TP': 0, 'TN': 0, 'FP': 0, 'FN': 0
    }
    time.sleep(1.0)

    try:
        # --- PHASE 1: BENIGN TRAFFIC (Ground Truth = 0) ---
        print(f"{CYAN}--- PHASE 1: STANDARD HOME TRAFFIC (Ground Truth: 0) ---{RESET}")
        for _ in range(15):
            if len(benign_pool) > 0:
                packet_features = benign_pool[np.random.randint(0, len(benign_pool))].reshape(1, -1)
                profile_name = "Empirical Benign"
            else:
                packet_features = np.array([[np.random.uniform(0.1, 0.8), np.random.uniform(0.05, 0.2), np.random.randint(200, 800), np.random.randint(1, 1000)]])
                profile_name = "Baseline Simulation"

            flow_key = (f"192.168.1.{np.random.randint(10, 50)}", "192.168.1.1", np.random.randint(1024, 65535), 443, "TCP")
            pred, prob, is_cached, lat = inspect_packet(model, packet_features, flow_key, flow_cache)

            stats['total'] += 1
            stats['algo_time_ns'] += (lat * 1000)
            if is_cached:
                stats['cached_skips'] += 1
                stats['cache_latencies'].append(lat)
            else:
                stats['ai_inferences'] += 1
                stats['ai_latencies'].append(lat)

            # Ground Truth Evaluation
            ground_truth = 0
            if pred == ground_truth:
                stats['TN'] += 1
                print(f"{GREEN}[ALLOW] Flow: {flow_key[0]}:{flow_key[2]:<5} | Status: BENIGN | Conf: {prob*100:5.1f}% | Lat: {lat:6.1f} μs{RESET}")
            else:
                stats['FP'] += 1
                print(f"{YELLOW}[WARNING] False Positive Detected!{RESET}")

            log_writer.writerow([datetime.now().isoformat(), str(flow_key), profile_name, ground_truth, pred, f"{prob:.4f}", is_cached, f"{lat:.2f}"])
            time.sleep(0.4)

        # --- PHASE 2: ATTACK TRAFFIC (Ground Truth = 1) ---
        print(f"\n{RED}!!! ANOMALY DETECTED: BOTNET FLOOD !!!{RESET}")
        print(f"{CYAN}--- PHASE 2: EMPIRICAL ATTACK REPLAY (Ground Truth: 1) ---{RESET}")
        time.sleep(1)

        botnet_nodes = [(f"185.22.44.{i}", np.random.randint(10000, 60000)) for i in range(1, 4)]

        for i in range(1, 101):
            b_ip, b_port = botnet_nodes[i % 3]
            packet_features = attack_pool[np.random.randint(0, len(attack_pool))].reshape(1, -1)
            flow_key = (b_ip, "192.168.1.1", b_port, 80, "TCP")

            pred, prob, is_cached, lat = inspect_packet(model, packet_features, flow_key, flow_cache)

            stats['total'] += 1
            stats['algo_time_ns'] += (lat * 1000)
            if is_cached:
                stats['cached_skips'] += 1
                stats['cache_latencies'].append(lat)
            else:
                stats['ai_inferences'] += 1
                stats['ai_latencies'].append(lat)

            # Ground Truth Evaluation
            ground_truth = 1
            if pred == ground_truth:
                stats['TP'] += 1
                if is_cached:
                    sys.stdout.write(f"{YELLOW}x{RESET}")
                    sys.stdout.flush()
                else:
                    print(f"\n{RED}[BLOCK] EMPIRICAL BOTNET SIGNATURE | Src: {b_ip:<15} | Conf: {prob*100:5.1f}%{RESET}")
            else:
                stats['FN'] += 1
                print(f"\n{YELLOW}[WARNING] False Negative - Attack Evaded!{RESET}")

            log_writer.writerow([datetime.now().isoformat(), str(flow_key), "Mirai Profile", ground_truth, pred, f"{prob:.4f}", is_cached, f"{lat:.2f}"])
            time.sleep(0.02)

        # --- PHASE 3: SCIENTIFIC DIAGNOSTIC REPORT ---
        reduction = (stats['cached_skips'] / stats['total']) * 100 if stats['total'] > 0 else 0
        avg_cache_lat = np.mean(stats['cache_latencies']) if stats['cache_latencies'] else 0
        avg_ai_lat = np.mean(stats['ai_latencies']) if stats['ai_latencies'] else 0

        algo_time_sec = stats['algo_time_ns'] / 1e9
        pps = stats['total'] / algo_time_sec if algo_time_sec > 0 else 0

        # Calculate Final Metrics
        TP, TN, FP, FN = stats['TP'], stats['TN'], stats['FP'], stats['FN']
        accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        print(f"\n\n{CYAN}==================================================")
        print("SIMULATION COMPLETE : CONFUSION MATRIX & METRICS")
        print(f"=================================================={RESET}")
        print(f"True Positives (Blocked) : {TP:<5} | False Positives : {FP}")
        print(f"True Negatives (Allowed) : {TN:<5} | False Negatives : {FN}")
        print(f"--------------------------------------------------")
        print(f"Accuracy                 : {accuracy*100:.2f}%")
        print(f"Precision                : {precision*100:.2f}%")
        print(f"Recall (Detection Rate)  : {recall*100:.2f}%")
        print(f"F1-Score                 : {f1:.4f}")
        print(f"--------------------------------------------------")
        print(f"Estimated Processing Thr : {pps:,.0f} packets/sec")
        print(f"AI Deep Inferences       : {stats['ai_inferences']} (Avg Python Latency: {avg_ai_lat:.1f} μs)")
        print(f"Flow Cache Lookups       : {stats['cached_skips']} (Avg Lookup Latency: {avg_cache_lat:.1f} μs)")
        print(f"Inference Reduction      : {GREEN}{reduction:.1f}% AI Workload Bypassed{RESET}")
        print(f"{CYAN}=================================================={RESET}")
        print(f"Complete ground-truth logs exported to: v10_evaluation_log.csv")

    except KeyboardInterrupt:
        pass
    finally:
        log_file.close()

if __name__ == "__main__":
    run_simulation()

V10 EDGE SHIELD : EDGE IDS EVALUATION PROTOTYPE
Architecture    : Decision Tree (Depth 5)
Training Date   : 2026-07-02 08:39:57.395625
Note            : Cache effectiveness is heavily dependent on flow reuse.

[SYSTEM] Loading empirical traffic from all CSV chunks...
[SUCCESS] Loaded 477 benign and 3668045 attack packets for replay.

--- PHASE 1: STANDARD HOME TRAFFIC (Ground Truth: 0) ---
[ALLOW] Flow: 192.168.1.38:39182 | Status: BENIGN | Conf:  66.7% | Lat: 1038.1 μs
[ALLOW] Flow: 192.168.1.17:45756 | Status: BENIGN | Conf:  98.5% | Lat: 1242.6 μs
[ALLOW] Flow: 192.168.1.48:7289  | Status: BENIGN | Conf:  98.5% | Lat: 1023.4 μs
[ALLOW] Flow: 192.168.1.32:38218 | Status: BENIGN | Conf:  98.5% | Lat: 1064.2 μs
[ALLOW] Flow: 192.168.1.33:61812 | Status: BENIGN | Conf:  98.5% | Lat:  986.8 μs
[ALLOW] Flow: 192.168.1.49:17047 | Status: BENIGN | Conf: 100.0% | Lat: 1034.2 μs
[ALLOW] Flow: 192.168.1.31:1793  | Status: BENIGN | Conf:  98.5% | Lat: 1108.7 μs
[ALLOW] Flow: 192.168.1.39:57125 